# KMX tracking checklist: what we can do today

This notebook works through the 11-item KMX tracking checklist (Slack DM, 15 Oct 2026). For each item it records:

- the data we hold now,
- a first cut where the data allows one, and
- what is still missing, and whether filling the gap needs scraping.

Coverage was checked against Snowflake on **2026-10-15**. Re-run the cells to refresh.

## Summary

| # | Item | Status today | Source | Gap / next step | Scrape needed? |
|---|---|---|---|---|---|
| 1 | CAF cumulative net loss by vintage | ✅ **Doable now** | `BOTS.AUTOABS.ABS_COMBINED_PROCESSED`, 20 CarMax trusts through **Aug-26** (reloaded 9 Oct) | No 2026 trusts (2026-1..4, 2026-A..C) | No. EDGAR is licensed and fair-access. 72degree or AutoABS should add the new trusts |
| 2 | Delinquency 30/60/90+ and roll rates | ✅ **Doable now** | Same table; `CURRENTDELINQUENCYSTATUS` = days past due | Same 2026-trust gap | No |
| 3 | Hidden credit stress | ✅ **Doable now** | `LOAN_EXTENDED`, `SUDDEN_PAID_OFF_WHILE_DELINQUENT60DAYS`, 90+ not repossessed, recovery rate | Same 2026-trust gap | No |
| 4 | Select scale and quality | 🟡 **Partly** | Pool FICO, PTI and LTV for Select 2024-A, 2025-A and 2025-B; Carvana N trusts for comparison | No 2026 Select trusts. Deal sizes from 424B/8-K are not loaded | No (EDGAR) |
| 5 | Markdowns / GPU proxy | 🟡 **Stale** | `YIPIT.USED_CAR.RETAILERS` (KMX) has initial vs sold price, but stops at **2026-07-06** | Ask Yipit why the feed stalled | No. Vendor ask |
| 6 | Price vs market and vs Carvana | 🟡 **Partly** | KMX vs CVNA on a like-for-like mix (Yipit retailers). KMX vs CarGurus sold price at make/model/year/mileage level | `YIPIT.CARG` in Snowflake has **no dealer field**, so there is no direct KMX-listing cut. Latest data is 31 Aug | **No.** Ask Yipit for a dealer/seller field. CarGurus restricts scraping |
| 7 | Wholesale prices and sourcing | 🟡 **Partly** | Slice CarMax appraisal emails (year/make/model/offer) | **No Manheim data in Snowflake.** Appraisal emails missing Dec-25 to May-26 | **Yes (Manheim).** Needs a Cox terms review in #bot-management first, so it was **not pulled here** |
| 8 | Off-lease supply | ❌ **No data** | — | Need ABS-EE for lease trusts | EDGAR (allowed). One-off script at `datapull/pull_lease_abs.py`. It could not run from this sandbox (sec.gov is blocked); run it locally |
| 9 | Omnichannel adoption | 🟡 **Partly** | Slice "Welcome to MyCarMax" (current to 14 Oct). Similarweb carmax.com traffic (pulled once, below) | Yipit 257-store delivery snapshot is **not in Snowflake**. Slice panel roughly halved from Jun-26 | No. Ask Yipit for the store-snapshot file |
| 10 | MaxCare attach | 🟡 **Broken since May-26** | Slice "Welcome to MaxCare" with term and miles, through **2026-04-20** | Zero emails from May-26: likely a parser or template change | No. Raise with Slice |
| 11 | Units, revenue and ASP | ✅ **Covered** | `MSCIENCE.KMX.SIGNALS`: MScience through 11 Oct, Yipit units/sales through 11 Oct (Yipit inventories through 31 Aug) | — | No |

**Scraping:** only item 7 (Manheim) needs a new scrape, and it needs a terms review first. Item 8 is a fair-access EDGAR pull. Every other gap is a vendor ask (Yipit, Slice) or loading new trusts.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import snowflake.connector

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

conn = snowflake.connector.connect(connection_name=os.getenv("SNOWFLAKE_CONNECTION", "default"))

def q(sql: str) -> pd.DataFrame:
    df = conn.cursor().execute(sql).fetch_pandas_all()
    df.columns = [c.lower() for c in df.columns]
    return df

ABS = "BOTS.AUTOABS.ABS_COMBINED_PROCESSED"
YIPIT_RETAILERS = "YIPIT.USED_CAR.RETAILERS"
CARG_SOLD = "YIPIT.CARG.DAILY_SOLD"
SIGNALS = "MSCIENCE.KMX.SIGNALS"
SLICE_ITEMS = "SRS_SLICE.COMMON.ITEMS_LIVE"

# Carvana 2021-N1..N4 are loaded twice: "2021-nX" (full history to Aug-26) and "2021-NX" (a reload
# covering Nov-21 to Dec-25 only). Keep the lowercase copy. RLIKE is case-sensitive.
ABS_DEDUP = "NOT RLIKE(trust_name, '.*-N[0-9]$')"

## 0. Freshness check

One row per source, so you can see at a glance what has gone stale since this notebook was written.

In [ ]:
q(f"""
SELECT 'ABS ' || source_table AS source, MAX(reportingperiodendingdate)::date AS latest, COUNT(DISTINCT trust_name) AS detail
FROM {ABS} WHERE source_table IN ('carmax', 'carvana') AND {ABS_DEDUP} GROUP BY 1
UNION ALL
SELECT 'Yipit retailers ' || retailer, MAX(date)::date, NULL FROM {YIPIT_RETAILERS} GROUP BY 1
UNION ALL
SELECT 'CarGurus sold (Yipit)', MAX(date)::date, NULL FROM {CARG_SOLD}
UNION ALL
SELECT datasource || ' ' || metric, MAX(date)::date, NULL FROM {SIGNALS} GROUP BY 1
UNION ALL
SELECT 'Slice CarMax: ' || CASE
         WHEN item_description ILIKE 'Welcome to MyCarMax%' THEN 'MyCarMax sign-up'
         WHEN item_description ILIKE 'Welcome to MaxCare%' THEN 'MaxCare welcome'
         WHEN item_description ILIKE '%appraisal value%' THEN 'appraisal offer'
         ELSE 'other' END,
       MAX(orderdate)::date, NULL
FROM {SLICE_ITEMS} WHERE merchant_name = 'CarMax' GROUP BY 1
ORDER BY 1
""")

## 1–3. CAF credit: cumulative net loss, delinquency and hidden stress

The `BOTS.AUTOABS` loan-level tables were reloaded on 9 Oct 2026 and now run through the **Aug-26** reporting period for both CarMax and Carvana. That resolves the "load new files" part of item 3. The 2026 CarMax trusts are still missing: the newest prime trust is 2025-4 and the newest Select trust is 2025-B.

Definitions (loan level, aggregated per trust and month on book):
- **CNL** = cumulative (charged-off principal − recoveries) ÷ pool balance at the first reporting period.
- **60+ DQ** = end balance of loans with `currentdelinquencystatus >= 60` days ÷ total end balance.
- Carvana 2021-N1..N4 appear twice under different-case trust names. The notebook keeps the complete copy (`ABS_DEDUP`). Worth fixing upstream.
- **Stress flags**: share of loans extended in the month, 90+ day loans not repossessed and not charged off, and loans that suddenly paid off while 60+ days delinquent.

In [ ]:
credit = q(f"""
WITH b AS (
    SELECT source_table, trust_name, prime_subprime,
           DATEDIFF('month', MIN(reportingperiodendingdate) OVER (PARTITION BY trust_name), reportingperiodendingdate) AS mob,
           reportingperiodendingdate AS period,
           chargedoffprincipalamount AS co, recoveredamount AS rec,
           reportingperiodbeginningloanbalanceamount AS beg, reportingperiodactualendbalanceamount AS endb,
           currentdelinquencystatus AS dq, loan_extended, sudden_paid_off_while_delinquent60days AS sudden_payoff,
           repossessedindicator AS repo
    FROM {ABS}
    WHERE source_table IN ('carmax', 'carvana') AND {ABS_DEDUP}
),
pool AS (SELECT trust_name, SUM(beg) AS orig_bal FROM b WHERE mob = 0 GROUP BY 1),
m AS (
    SELECT source_table, trust_name, prime_subprime, mob, MAX(period) AS period,
           SUM(co) AS co, SUM(rec) AS rec, SUM(endb) AS endb,
           SUM(IFF(dq >= 30, endb, 0)) AS dq30, SUM(IFF(dq >= 60, endb, 0)) AS dq60, SUM(IFF(dq >= 90, endb, 0)) AS dq90,
           COUNT(*) AS loans, COUNT_IF(loan_extended = 1) AS extended,
           COUNT_IF(dq >= 90 AND NOT COALESCE(repo, FALSE) AND COALESCE(co, 0) = 0) AS dq90_not_repo,
           COUNT_IF(sudden_payoff = 1) AS sudden_payoff
    FROM b GROUP BY 1, 2, 3, 4
)
SELECT m.*, p.orig_bal,
       SUM(co - rec) OVER (PARTITION BY m.trust_name ORDER BY mob) / p.orig_bal AS cnl,
       SUM(rec) OVER (PARTITION BY m.trust_name ORDER BY mob) / NULLIF(SUM(co) OVER (PARTITION BY m.trust_name ORDER BY mob), 0) AS cum_recovery_rate,
       dq30 / NULLIF(endb, 0) AS dq30_pct, dq60 / NULLIF(endb, 0) AS dq60_pct, dq90 / NULLIF(endb, 0) AS dq90_pct,
       extended / loans AS ext_rate, dq90_not_repo / loans AS dq90_not_repo_rate, sudden_payoff / loans AS sudden_payoff_rate
FROM m JOIN pool p USING (trust_name)
ORDER BY trust_name, mob
""")
credit["vintage"] = credit.trust_name.str.extract(r"(\d{4}-\w+)$")[0]
credit["year"] = credit.vintage.str[:4]
credit.groupby(["source_table", "prime_subprime"]).agg(trusts=("trust_name", "nunique"), latest=("period", "max"))

In [ ]:
def curve_grid(metric, title, max_mob=48):
    segs = [("carmax", "Prime"), ("carmax", "Subprime"), ("carvana", "Prime"), ("carvana", "Subprime")]
    fig, axes = plt.subplots(1, 4, figsize=(20, 4.5), sharey=True)
    for ax, (src, tier) in zip(axes, segs):
        d = credit[(credit.source_table == src) & (credit.prime_subprime == tier) & (credit.mob <= max_mob)]
        for (trust, yr), g in d.groupby(["trust_name", "year"]):
            ax.plot(g.mob, g[metric], lw=1, label=trust.split()[-1], color=plt.cm.viridis((int(yr) - 2020) / 7))
        ax.set_title(f"{src} {tier}"); ax.set_xlabel("months on book"); ax.grid(alpha=0.3)
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1%}"))
        if len(d): ax.legend(fontsize=6, ncol=2)
    fig.suptitle(title); plt.tight_layout(); plt.show()

curve_grid("cnl", "1. Cumulative net loss by vintage")
curve_grid("dq60_pct", "2. 60+ day delinquency (% of end balance)")

In [ ]:
# Same-age comparison: CNL and 60+ DQ at fixed months on book, by trust (CarMax only)
kmx = credit[credit.source_table == "carmax"]
pd.concat({
    "cnl": kmx[kmx.mob.isin([6, 12, 18, 24, 36])].pivot(index="trust_name", columns="mob", values="cnl"),
    "dq60": kmx[kmx.mob.isin([6, 12, 18, 24, 36])].pivot(index="trust_name", columns="mob", values="dq60_pct"),
}, axis=1).style.format("{:.2%}", na_rep="")

In [ ]:
# 2b. Roll rates: share of balance moving from 30-59 DPD to 60+ DPD the next month (CarMax, by tier)
rolls = q(f"""
SELECT prime_subprime, reportingperiodendingdate::date AS period,
       SUM(IFF(prev_currentdelinquencystatus BETWEEN 30 AND 59 AND currentdelinquencystatus >= 60, reportingperiodactualendbalanceamount, 0))
         / NULLIF(SUM(IFF(prev_currentdelinquencystatus BETWEEN 30 AND 59, prev_reportingperiodactualendbalanceamount, 0)), 0) AS roll_30_to_60,
       SUM(IFF(prev_currentdelinquencystatus = 0 AND currentdelinquencystatus >= 30, reportingperiodactualendbalanceamount, 0))
         / NULLIF(SUM(IFF(prev_currentdelinquencystatus = 0, prev_reportingperiodactualendbalanceamount, 0)), 0) AS roll_current_to_30
FROM {ABS}
WHERE source_table = 'carmax' AND reportingperiodendingdate >= '2023-01-01'
GROUP BY 1, 2 ORDER BY 1, 2
""")
rolls.pivot(index="period", columns="prime_subprime", values=["roll_current_to_30", "roll_30_to_60"]).plot(figsize=(12, 4), title="CarMax roll rates")
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1%}")); plt.show()

In [ ]:
# 3. Hidden-stress flags by calendar month, CarMax prime vs Select
stress = (credit[credit.source_table == "carmax"]
          .groupby(["period", "prime_subprime"])[["extended", "dq90_not_repo", "sudden_payoff", "loans"]].sum())
stress = stress[["extended", "dq90_not_repo", "sudden_payoff"]].div(stress.loans, axis=0).unstack()
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, col, title in zip(axes, ["extended", "dq90_not_repo", "sudden_payoff"],
                          ["Loans extended this month", "90+ DPD, not repossessed or charged off", "Sudden payoff while 60+ DPD"]):
    stress[col].plot(ax=ax, title=title); ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.2%}")); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

curve_grid("cum_recovery_rate", "3. Cumulative recovery rate (recoveries / charge-offs)")

**Still to do for 1–3:**
- Load the 2026 CarMax trusts. Christopher flagged the `bamsec_detail_` → `carmax_detail_` file-name filter in the AutoABS config.
- Overlay CAF's 3.02% loss provision on the CNL chart.
- Schedule the refresh. 72degree delivers monthly, around the 16th.

## 4. CarMax Select: scale and pool quality

Pool composition at the first reporting period, from loan-level data. Carvana N trusts are included for comparison. Deal-size and issuance cadence from 424B5 and 8-K filings are **not loaded**. They are small EDGAR pulls, or can be read from the `CARMAX_SEC2_LIST` filing index.

In [ ]:
pools = q(f"""
WITH first AS (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY trust_name, assetnumber ORDER BY reportingperiodendingdate) AS rn,
           MIN(reportingperiodendingdate) OVER (PARTITION BY trust_name) AS first_period
    FROM {ABS}
    WHERE source_table IN ('carmax', 'carvana') AND prime_subprime = 'Subprime' AND {ABS_DEDUP}
)
SELECT source_table, trust_name, first_period::date AS first_period,
       COUNT(*) AS loans,
       SUM(reportingperiodbeginningloanbalanceamount) / 1e6 AS pool_bal_mm,
       SUM(obligorcreditscore * reportingperiodbeginningloanbalanceamount) / SUM(IFF(obligorcreditscore > 0, reportingperiodbeginningloanbalanceamount, 0)) AS wa_fico,
       COUNT_IF(obligorcreditscore IS NULL OR obligorcreditscore = 0) / COUNT(*) AS no_fico_share,
       SUM(paymenttoincomepercentage * reportingperiodbeginningloanbalanceamount) / SUM(reportingperiodbeginningloanbalanceamount) AS wa_pti,
       SUM(ltv_ratio * reportingperiodbeginningloanbalanceamount) / SUM(reportingperiodbeginningloanbalanceamount) AS wa_ltv,
       SUM(originalinterestratepercentage * reportingperiodbeginningloanbalanceamount) / SUM(reportingperiodbeginningloanbalanceamount) AS wa_apr
FROM first WHERE rn = 1 AND reportingperiodendingdate = first_period
GROUP BY 1, 2, 3 ORDER BY 3
""")
pools.style.format({"pool_bal_mm": "{:,.0f}", "wa_fico": "{:.0f}", "no_fico_share": "{:.1%}", "wa_pti": "{:.3f}", "wa_ltv": "{:.3f}", "wa_apr": "{:.3f}"})

## 5–6. Pricing: markdowns, price vs Carvana, price vs market

**Item 5.** `YIPIT.USED_CAR.RETAILERS` has KMX initial and sold price at make/model/year/mileage-band level. Discount = 1 − sold ÷ initial. **The KMX rows stop at 2026-07-06**, while CVNA runs to 11 Oct. Ask Yipit why the KMX feed stalled.

**Item 6.** The checklist assumed we could cut CarMax listings out of `YIPIT.CARG`. In Snowflake, the CarGurus tables (`DAILY_LISTINGS`, `DAILY_SOLD`, `DAILY_AGG`) carry **no dealer or seller field**. They are aggregated by source, state, make, model, trim, year and mileage band. Two proxies are possible today:
- **KMX vs CVNA:** price each month's KMX sold mix at CVNA's sold price for the same make/model/year/mileage cell.
- **KMX vs market:** price the same KMX mix at the CarGurus US used sold price for the cell. This compares against all CarGurus dealers rather than CarGurus' "expected price", so read it as a trend, not a level.

The fix is a vendor ask, not a scrape: ask Yipit to add a seller/dealer field (or a CarMax flag) to the CARG delivery, plus weekly cadence.

In [ ]:
px = q(f"""
WITH r AS (
    SELECT DATE_TRUNC('month', date)::date AS m, retailer, make_unified AS mk, model_unified AS md, year, mileage_banc AS mb,
           SUM(units) AS u, SUM(sold_price_total) AS sp, SUM(initial_price_total) AS ip
    FROM {YIPIT_RETAILERS} WHERE date >= '2024-01-01' GROUP BY 1, 2, 3, 4, 5, 6
),
k AS (SELECT * FROM r WHERE retailer = 'KMX'),
c AS (SELECT * FROM r WHERE retailer = 'CVNA'),
cg AS (
    SELECT DATE_TRUNC('month', date)::date AS m, UPPER(make) AS mk, UPPER(model) AS md, year, mileage_banc AS mb,
           SUM(price * inventory) / NULLIF(SUM(inventory), 0) AS cg_price
    FROM {CARG_SOLD} WHERE source = 'www.cargurus.com' AND inventory_type = 'used' GROUP BY 1, 2, 3, 4, 5
),
kmx_all AS (SELECT m, SUM(u) AS kmx_units, 1 - SUM(sp) / SUM(ip) AS kmx_discount FROM k GROUP BY 1),
cvna_all AS (SELECT m, 1 - SUM(sp) / SUM(ip) AS cvna_discount FROM c GROUP BY 1),
matched AS (
    SELECT k.m,
           SUM(k.sp) / SUM(k.u * c.sp / c.u) - 1 AS kmx_vs_cvna_same_mix,
           SUM(IFF(cg.cg_price IS NOT NULL, k.sp, 0)) / NULLIF(SUM(IFF(cg.cg_price IS NOT NULL, k.u * cg.cg_price, 0)), 0) - 1 AS kmx_vs_cargurus_same_mix,
           SUM(IFF(cg.cg_price IS NOT NULL, k.u, 0)) / SUM(k.u) AS cargurus_match_share
    FROM k JOIN c USING (m, mk, md, year, mb)
    LEFT JOIN cg ON cg.m = k.m AND cg.mk = UPPER(k.mk) AND cg.md = UPPER(k.md) AND cg.year = k.year AND cg.mb = k.mb
    GROUP BY 1
)
SELECT * FROM kmx_all LEFT JOIN cvna_all USING (m) LEFT JOIN matched USING (m) ORDER BY m
""").set_index("m")

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
px[["kmx_discount", "cvna_discount"]].plot(ax=axes[0], title="5. Discount from initial list price (Yipit)")
px[["kmx_vs_cvna_same_mix", "kmx_vs_cargurus_same_mix"]].plot(ax=axes[1], title="6. KMX sold price premium, same mix")
for ax in axes: ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1%}")); ax.grid(alpha=0.3); ax.axhline(0, color="k", lw=0.5)
plt.tight_layout(); plt.show()
px.tail(15).style.format({"kmx_units": "{:,.0f}", **{c: "{:.2%}" for c in px.columns if c != "kmx_units"}})

Note: the last month in the KMX series (Jul-26) is partial because the feed stops on 6 Jul.

## 7. Wholesale prices and vehicle sourcing

- **Manheim Used Vehicle Value Index: not available.** It is not in Snowflake. Cox publishes it free on manheim.com, but the checklist says Cox's terms must be reviewed in #bot-management before we pull it, so **no pull was made here**. Once cleared, it is a twice-monthly fetch of one headline number, and a manual copy is a good stopgap.
- **Sourcing: Slice CarMax instant-offer appraisal emails** (`"<year> <make> <model>: appraisal value USD <n>"`). The daily feed has year, make, model and offer value. There are **no appraisal emails from Dec-25 to May-26**, and the Slice CarMax panel has shrunk by about half since Jun-26. Normalise per active mailbox before reading trends.

In [ ]:
slice_kmx = q(f"""
SELECT DATE_TRUNC('month', orderdate)::date AS month,
       COUNT_IF(item_description ILIKE 'Welcome to MyCarMax%') AS mycarmax_signups,
       COUNT_IF(item_description ILIKE 'Welcome to MaxCare%') AS maxcare_welcomes,
       COUNT_IF(item_description ILIKE '%appraisal value%') AS appraisals,
       COUNT_IF(item_description = 'Payment') AS payments,
       COUNT(DISTINCT mailbox_id) AS active_mailboxes
FROM {SLICE_ITEMS}
WHERE merchant_name = 'CarMax' AND orderdate >= '2023-01-01'
GROUP BY 1 ORDER BY 1
""").set_index("month")
slice_kmx

In [ ]:
appr = q(f"""
SELECT orderdate::date AS date,
       TRY_TO_NUMBER(REGEXP_SUBSTR(item_description, '^([0-9]{{4}}) ', 1, 1, 'e', 1)) AS model_year,
       REGEXP_SUBSTR(item_description, '^[0-9]{{4}} ([^:]+):', 1, 1, 'e', 1) AS make_model,
       TRY_TO_NUMBER(REPLACE(REGEXP_SUBSTR(item_description, 'USD ([0-9,]+)', 1, 1, 'e', 1), ',', '')) AS offer_usd
FROM {SLICE_ITEMS}
WHERE merchant_name = 'CarMax' AND item_description ILIKE '%appraisal value%' AND orderdate >= '2023-01-01'
""")
appr["month"] = pd.to_datetime(appr.date).dt.to_period("M")
appr["vehicle_age"] = pd.to_datetime(appr.date).dt.year - appr.model_year
m = appr[appr.vehicle_age.between(1, 8)].groupby("month").agg(offers=("offer_usd", "size"), median_offer=("offer_usd", "median"), median_age=("vehicle_age", "median"))
m["median_offer"].plot(figsize=(12, 4), marker="o", title="7. Median CarMax instant offer, 1–8 yr old vehicles (Slice)"); plt.grid(alpha=0.3); plt.show()
m.tail(18)

## 8. Off-lease supply

**No data held.** Public auto-lease trusts file ABS-EE (EX-102) with each lease's maturity date and residual value. The one-off script `KMX/datapull/pull_lease_abs.py` pulls the latest ABS-EE for each target trust and builds a forward maturity schedule (count and residual value by month). It follows SEC fair-access rules: a declared User-Agent and 5 requests/sec or fewer, the same pattern as `carmax/datapull/pull_carmax_abs.py`.

It **has not been run**, because the sandbox used to write this notebook cannot reach sec.gov. Run it locally:

```
python KMX/datapull/pull_lease_abs.py
```

The output goes to `KMX/datapull/output/lease_maturity_schedule.csv` (gitignored). The cell below reads it if it exists.

In [ ]:
from pathlib import Path
sched_path = Path("datapull/output/lease_maturity_schedule.csv")
if sched_path.exists():
    sched = pd.read_csv(sched_path, parse_dates=["maturity_month"])
    piv = sched.pivot_table(index="maturity_month", columns="issuer", values="leases", aggfunc="sum").fillna(0)
    piv[piv.index >= pd.Timestamp.today() - pd.DateOffset(months=6)].plot.area(figsize=(14, 4), title="8. Scheduled lease maturities, public lease ABS")
    plt.show()
else:
    print("No lease schedule yet: run KMX/datapull/pull_lease_abs.py")

## 9. Omnichannel adoption

- **Yipit 257-store delivery/pickup snapshot: not in Snowflake.** No table matches it. Ask Yipit for the file or have it added to the delivery.
- **MyCarMax sign-ups (Slice)** are current to 14 Oct. They are in the `slice_kmx` table above. Divide by `active_mailboxes`, because the panel roughly halved from Jun-26.
- **carmax.com traffic (Similarweb)** was pulled once on 2026-10-15 via the Similarweb connector (US, all devices, monthly) and is pasted below. To refresh, re-pull via the connector, or ask for carmax.com to be added to the `SIMILARWEB` Snowflake share. Today the share holds only NFLX, SHOP and TSLA.

In [ ]:
# Similarweb, US total traffic, millions of visits / unique visitors. One-off pull 2026-10-15.
sw = pd.DataFrame({
    "month": pd.period_range("2024-01", "2026-09", freq="M"),
    "carmax_visits_m": [19.31, 19.46, 20.88, 19.73, 20.52, 19.53, 20.04, 19.69, 16.50, 15.24, 13.91, 15.04,
                        15.58, 14.99, 17.46, 15.95, 15.95, 15.92, 17.65, 17.79, 15.45, 15.03, 14.43, 15.45,
                        18.10, 19.06, 22.56, 21.97, 22.68, 20.39, 19.89, 20.88, 17.22],
    "carmax_uv_m":     [9.03, 9.03, 9.40, 9.35, 9.68, 9.10, 9.13, 9.02, 7.42, 6.91, 6.63, 7.25,
                        7.61, 7.21, 8.24, 7.45, 7.71, 7.70, 8.45, 8.40, 7.58, 7.26, 6.97, 7.52,
                        8.91, 9.34, 10.46, 10.67, 10.85, 9.77, 9.82, 10.05, 8.31],
    "carvana_visits_m": [18.19, 17.40, 19.36, 20.04, 21.71, 21.41, 21.93, 22.40, 19.66, 19.96, 18.96, 19.60,
                         20.84, 20.80, 22.65, 19.75, 21.45, 23.75, 25.72, 25.62, 23.19, 24.39, 24.05, 26.23,
                         27.76, 26.33, 30.32, 31.30, 34.44, 39.09, 41.61, 37.04, 33.37],
    "carvana_uv_m":     [9.18, 8.72, 9.84, 9.97, 10.96, 11.01, 11.25, 11.34, 9.80, 9.77, 9.43, 9.68,
                         10.46, 10.51, 11.42, 10.44, 11.09, 12.54, 13.45, 13.38, 11.93, 12.49, 12.47, 13.61,
                         14.29, 13.39, 14.60, 15.74, 16.56, 17.78, 20.55, 18.58, 17.18],
}).set_index("month")
sw["carmax_visit_share"] = sw.carmax_visits_m / (sw.carmax_visits_m + sw.carvana_visits_m)
sw["carmax_visits_yoy"] = sw.carmax_visits_m.pct_change(12)

omni = slice_kmx.copy()
omni.index = pd.PeriodIndex(omni.index, freq="M")
omni["mycarmax_per_1k_mailboxes"] = omni.mycarmax_signups / omni.active_mailboxes * 1000

fig, axes = plt.subplots(1, 3, figsize=(20, 4))
sw[["carmax_visits_m", "carvana_visits_m"]].plot(ax=axes[0], title="Monthly US visits (m), Similarweb")
sw["carmax_visit_share"].plot(ax=axes[1], title="CarMax share of KMX+CVNA visits")
omni["mycarmax_per_1k_mailboxes"].plot(ax=axes[2], title="MyCarMax sign-ups per 1k active Slice mailboxes")
for ax in axes: ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
sw.tail(12).round(3)

## 10. MaxCare (extended service plan) attach

The Slice "Welcome to MaxCare <term> months or <miles> miles" emails give plan term and mileage, but **they stop on 2026-04-20**. There are zero from May-26, while other CarMax emails continue, so this is likely a template or parser change on Slice's side. Raise it with Slice.

Attach proxy: MaxCare welcomes ÷ CarMax purchase signals in the same panel. Neither Slice series is a clean count of car purchases, so treat this as a trend only.

In [ ]:
mc = q(f"""
SELECT DATE_TRUNC('month', orderdate)::date AS month,
       TRY_TO_NUMBER(REGEXP_SUBSTR(item_description, 'MaxCare ([0-9]+) months', 1, 1, 'e', 1)) AS term_months,
       TRY_TO_NUMBER(REPLACE(REGEXP_SUBSTR(item_description, 'or ([0-9,]+) miles', 1, 1, 'e', 1), ',', '')) AS miles,
       COUNT(*) AS plans
FROM {SLICE_ITEMS}
WHERE merchant_name = 'CarMax' AND item_description ILIKE 'Welcome to MaxCare%' AND orderdate >= '2023-01-01'
GROUP BY 1, 2, 3
""")
mix = mc.dropna(subset=["term_months"]).pivot_table(index="month", columns="term_months", values="plans", aggfunc="sum").fillna(0)
mix.div(mix.sum(axis=1), axis=0).plot.area(figsize=(12, 4), title="10. MaxCare plan term mix (Slice)"); plt.show()

attach = slice_kmx[["maxcare_welcomes", "mycarmax_signups", "active_mailboxes"]].copy()
attach["maxcare_per_1k_mailboxes"] = attach.maxcare_welcomes / attach.active_mailboxes * 1000
attach.tail(24)

## 11. Units, revenue and ASP (baseline)

This is already covered by the `sync_carmax` workflow. MScience units, sales, inventories and stores run through 11 Oct. Yipit units and sales also run through 11 Oct, with inventories through 31 Aug. Quarterly KPIs and consensus are in `MSCIENCE.KMX.KPIS`. Yipit publishes several panel-method variants (modelled and observed, daily and monthly), so each is shown as its own y/y series.

In [ ]:
units = q(f"""
WITH d AS (  -- one value per source, panel method, day and state (Yipit repeats some rows)
    SELECT datasource || ' / ' || panel_method AS series, date, attr_value, MAX(metric_value) AS v
    FROM {SIGNALS}
    WHERE metric = 'UNITS' AND date >= DATEADD('year', -2, CURRENT_DATE)
    GROUP BY 1, 2, 3
)
SELECT series, DATE_TRUNC('week', date)::date AS week, SUM(v) AS units
FROM d GROUP BY 1, 2 ORDER BY 2
""")
u = units.pivot(index="week", columns="series", values="units")
(u / u.shift(52) - 1).iloc[52:].plot(figsize=(12, 4), title="11. Weekly KMX units, y/y"); plt.grid(alpha=0.3)
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}")); plt.show()

## Actions

| Owner | Ask | Unblocks |
|---|---|---|
| AutoABS / 72degree | Load the 2026 CarMax trusts (file-name filter fix) | 1–4 |
| Yipit | Why did the KMX retailer feed stop on 6 Jul? Can it be weekly? | 5, 6 |
| Yipit | Add a dealer/seller field (or CarMax flag) to the CARG delivery. Send the 257-store omnichannel snapshot | 6, 9 |
| Slice | MaxCare welcome emails are missing since May-26. Appraisal emails had a Dec-25 to May-26 gap. Why has the panel shrunk? | 7, 9, 10 |
| #bot-management | Terms review for the Cox/Manheim index pull | 7 |
| Me | Run `datapull/pull_lease_abs.py` locally (EDGAR, fair access) | 8 |
| Similarweb | Add carmax.com and carvana.com to the Snowflake share, or schedule a monthly connector pull | 9 |